# PD / ET / N Separation — condition-aware experiments

Lives in **`pdetn/`**, separate from the deep pipeline in `tremor/`. Builds **one
condition-aware feature vector per patient** (interpretable spectral biomarkers for
REST/OUT/WING + rest-vs-action contrasts) and evaluates **flat** vs **two-stage**
(N-vs-tremor, then PD-vs-ET) classifiers under honest **leave-one-patient-out** with
subject bootstrap CIs. CPU-only. Section 7 compares against the **deep** STFT/CWT/HHT
models.

## 1. Setup

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "pdetn":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, subprocess, json
import matplotlib.pyplot as plt
from tremor.data import CLASS_NAMES
from pdetn.features import CONDITIONS
from pdetn.model import FlatClassifier, TwoStageClassifier
from pdetn.evaluate import evaluate, print_result
from pdetn.run import load_features
print("classes:", CLASS_NAMES, "| conditions:", CONDITIONS)

## 2. Load data & build per-patient features

In [ ]:
DATA_ROOT = "Data"   # <-- change if needed
X, y, subjects, names = load_features(DATA_ROOT, CONDITIONS)
import collections
print(f"patients: {len(subjects)}  features: {X.shape[1]}")
print("class counts:", {CLASS_NAMES[k]: v for k,v in sorted(collections.Counter(y).items())})

## 3. Flat 3-class baseline\n`estimator`: `'logreg'` (fast, best here) | `'lda'` | `'rf'` (strong but slow, tends to collapse ET).

In [ ]:
estimator = "logreg"
res_flat = evaluate(lambda: FlatClassifier(estimator), X, y, subjects, n_boot=1000)
print_result(f"FLAT ({estimator})", res_flat)

## 4. Two-stage with tuned ET threshold\nN-vs-tremor, then PD-vs-ET. The ET probability threshold is tuned by internal CV on the training fold (leakage-free) — this is what stops ET from collapsing under the PD majority.

In [ ]:
res_two = evaluate(lambda: TwoStageClassifier(estimator, estimator, tune_et_threshold=True),
                   X, y, subjects, n_boot=1000)
print_result(f"TWO-STAGE tuned ({estimator})", res_two)

## 5. Confusion matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, res, ttl in [(ax[0], res_flat, "Flat"), (ax[1], res_two, "Two-stage tuned")]:
    ConfusionMatrixDisplay(np.array(res["confusion_matrix"]), display_labels=CLASS_NAMES).plot(ax=a, colorbar=False)
    a.set_title(f"{ttl}  (macro-F1 {res['macro_f1']:.2f}, ET-F1 {res['per_class_f1']['ET']:.2f})")
plt.tight_layout(); plt.show()

## 6. Compare estimators (two-stage, tuned)

In [ ]:
rows=[]
for est in ["logreg","lda","rf"]:
    r=evaluate(lambda e=est: TwoStageClassifier(e,e,tune_et_threshold=True), X,y,subjects, n_boot=500)
    ci=r["ci"]["ET"]; rows.append((f"pdetn 2stage {est}", r["macro_f1"], r["per_class_f1"]["ET"], ci["lo"], ci["hi"]))
    print(f"{est:>7}: macroF1={r['macro_f1']:.3f}  ET-F1={r['per_class_f1']['ET']:.3f} [{ci['lo']:.2f},{ci['hi']:.2f}]")
pdetn_rows = rows

## 7. Compare with the DEEP models (STFT / CWT / HHT / wavelet)

Two ways:
* **Reference numbers** measured this session (single seed 42, same ET-LOSO + in-CV threshold + bootstrap CI) — instant.
* **Re-run live** via `run_deep(method)` which calls `tremor.cv_benchmark` (slow: STFT/CWT ~3–5 min; `wavelet_packet`/`multitaper` similar; **HHT is ~100× slower and impractical on CPU** — run on GPU).

In [ ]:
# Reference deep results measured this session (OUT, seed 42, threshold-tuned)
deep_reference = {
    "deep STFT":        {"macro_f1": 0.658, "ET_f1": 0.571, "ET_auc": 0.78},
    "deep CWT":         {"macro_f1": 0.652, "ET_f1": 0.545, "ET_auc": 0.77},
    "deep wavelet_pkt": {"macro_f1": 0.618, "ET_f1": 0.372, "ET_auc": 0.76},
    "deep multitaper":  {"macro_f1": 0.611, "ET_f1": 0.468, "ET_auc": 0.75},
    # HHT (EMD) not run: ~100x slower per-epoch on CPU. Use --hht-emd-method on GPU.
}
for k,v in deep_reference.items():
    print(f"{k:>16}: macroF1={v['macro_f1']:.3f}  ET-F1(thresh)={v['ET_f1']:.3f}  ET-AUC={v['ET_auc']:.2f}")

In [ ]:
def run_deep(method="stft", data_root="Data", epochs=80, seed=42, out="artifacts/nb_deep"):
    """Run the deep ET-LOSO for one TFD method and return its pooled metrics.
    Slow: stft/cwt ~3-5 min on CPU; hht impractical. Requires torch installed."""
    odir = f"{out}/{method}"
    cmd = ["python","-m","tremor.cv_benchmark","--data-root",data_root,"--action","OUT",
           "--data-mode","quaternion","--model","tremor_bilstm",
           "--quaternion-sensors","hand,lower_arm,upper_arm","--tfd-methods",method,
           "--f-max","15","--normalize","per_recording","--loss","focal","--focal-gamma","1.5",
           "--oversample-to","-1","--loso-target-class","ET","--tune-et-threshold",
           "--epochs",str(epochs),"--patience","15","--n-boot","1000","--n-perm","1000",
           "--seed",str(seed),"--output",odir]
    subprocess.run(cmd, check=True)
    rep = json.load(open(f"{odir}/{method}/pooled_report.json"))
    et = rep.get("et_threshold",{}).get("thresholded",{}).get("per_class_f1",{}).get("ET",{})
    return {"macro_f1": rep["macro_f1"], "ET_f1_argmax": rep["per_class"]["ET"]["f1"],
            "ET_f1_thresh": et.get("point"), "ET_auc": rep["per_class"]["ET"]["auc"]}
# Example (uncomment to run live, ~4 min):
# print(run_deep("stft"))

### Combined comparison chart — pdetn vs deep

In [ ]:
labels, macro, etf1 = [], [], []
for name, m, et, lo, hi in pdetn_rows:
    labels.append(name); macro.append(m); etf1.append(et)
for name, v in deep_reference.items():
    labels.append(name); macro.append(v["macro_f1"]); etf1.append(v["ET_f1"])
xi = np.arange(len(labels)); w = 0.4
fig, ax = plt.subplots(figsize=(11,4))
ax.bar(xi-w/2, macro, w, label="macro-F1", color="#2c7fb8")
ax.bar(xi+w/2, etf1, w, label="ET-F1", color="#d95f02")
ax.set_xticks(xi); ax.set_xticklabels(labels, rotation=30, ha="right")
ax.axhline(1/3, ls="--", c="gray", lw=1, label="chance (3-class)")
ax.set_ylabel("F1"); ax.set_title("pdetn (interpretable) vs deep models"); ax.legend()
plt.tight_layout(); plt.show()

## 8. Notes

- **N-vs-tremor** is easy; **PD-vs-ET** is the ceiling. The tuned ET threshold is what keeps ET from collapsing (RF collapses it to F1=0 without help).
- The interpretable two-stage (`logreg`, ~macro-F1 0.58 / ET-F1 0.32) trails the deep STFT model (~0.66 / 0.57) but is transparent and CPU-only — a good ablation/interpretability arm for the paper.
- On ~16 ET patients, differences among methods largely fall **inside the CIs**; the real lever is **external ET data (PADS, 16→44)** — see `reports/track3_external_data.md`.
- HHT with EEMD/CEEMDAN and pretrained AST/ResNet18 need a **GPU + internet** machine; the code supports them (`--hht-emd-method`, `--model ast|resnet18`).